## Google Colab, Mediapipe를 사용하여 mp4 비디오에서 Landmark 추출하기
* mp4를 다수개 저장한 Google Drive의 경로와 csv 경로를 지정해주고 아래의 코드를 실행하면 된다
* 이 코드를 실행하기 전에 mp4 비디오가 준비되어 Google Drive에 업로드된 상태이어야 한다
* 실제 비디오, Blender 애니메이션에서 파생된 비디오를 대상으로 Landmark를 추출
* Landmark를 이용하여 LSTM 모델 생성/학습/행동인식 테스트 기능 포함

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install mediapipe==0.10.18

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 22.1 MB/s eta 0:00:00
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.20
    Uninstalling mediapipe-0.10.20:
      Successfully uninstalled mediapipe-0.10.20


In [ ]:
!pip install mediapipe
import mediapipe as mp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 36.0 MB/s eta 0:00:00


## Google Mediapipe를 사용하여 비디오(mp4)에서 관절좌표(Landmarks) 추출하고 CSV에 저장
* 실제 비디오(mp4) 파일이나 Blender에서 렌더링된 mp4 파일로부터 Landmarks를 추출할 때 사용
* 디렉토리에서 포함된 모든 mp4를 로드하고 관절 좌표 추출
* Google Drive에 walk, run 디렉토리를 생성하고 각각 mp4 비디오를 저장한다
* 비디오에서 추출된 Landmark가 저장될 디렉토리를 Google Drive에 지정해야 한다

In [ ]:
# 다수 개의 mp4 비디오가 포함된 디렉토리에서 각각의 비디오로부터 Landmark를 추출하여 각 csv파일에 저장한다
# 걷기/달리기 비디오(mp4)에서 각 프레임 안에 포함된 관절의 위치를 추출하여 사본 비디오에 표시하고 관절 좌표를 csv 파일에 저장하는 예
# 프레임 영상을 화면에 표시하지 않고 영상 위의 관절에 점을 표시하지도 않으며 단지 관절 좌표를 CSV 파일에 저장하기만 함
import cv2
import os
from google.colab.patches import cv2_imshow
import mediapipe as mp
import numpy as np
import sys
import pandas as pd

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
#print(pose)   # mediapipe.python.solutions.pose.Pose

#다수개의 비디오(mp4)를 읽어서 비디오의 각 프레임에 포함된 landmark 정보를 추출하는 반복 작업
tmp_col = '''LS_x, LS_y, LS_z, RS_x, RS_y, RS_z, LFA_x, LFA_y, LFA_z, RFA_x, RFA_y, RFA_z, LH_x, LH_y, LH_z, RH_x, RH_y, RH_z,
            LUL_x,  LUL_y, LUL_z, RUL_x, RUL_y, RUL_z, LL_x, LL_y, LL_z, RL_x, RL_y, RL_z, LF_x, LF_y, LF_z, RF_x, RF_y, RF_z'''
columns = [ tok.strip() for tok in tmp_col.split(',')]
#df = pd.DataFrame( columns=columns)

video_directory = '/content/drive/MyDrive/Thesis/Anim_videos/Real_Videos/run/'
no_landmark_frames = 0  # 한개의 비디오에서 landmark를 추출할 수 없는 프레임의 수
failed_file_names = []

for entry in os.listdir(video_directory):
    if os.path.isfile(os.path.join(video_directory, entry)) and entry.lower().endswith(".mp4"):
        video_path = os.path.join(video_directory, entry)
        print('비디오 파일명-> ', video_path)

        cap = cv2.VideoCapture(video_path)

        no_landmark_frames = 0  # 한개의 비디오에서 landmark를 추출할 수 없는 프레임의 수

        #check if the video capture is open
        if(cap.isOpened() == False):
            print("Error Opening Video Stream Or File")

        w = round(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = round(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        #fourcc = cv2.VideoWriter_fourcc(*'DIVX')

        # 1프레임과 다음 프레임 사이의 간격 설정
        delay = round(1000/fps)

        # 영상 저장
        # cv2.VideoWriter 객체 생성, 속성값 입력
        #out = cv2.VideoWriter('/content/drive/MyDrive/Thesis/run_0001-0067_output.mp4', fourcc, 20.0, (w, h))    # 비디오파일 저장을 위한 스트림 생성

        # landmark를 csv 파일에 저장하기 위한 자료구조
        landmark_list = []  # 한개의 프레임에 포함된 관절좌표,visibility를 한행으로 표현(2차원 배열)

        frame_cnt = 0

        bone_idx = [11,12,13,14,15,16,23,24,25,25,27,28]

        df = pd.DataFrame( columns=columns)

        while(cap.isOpened()):
            ret, frame = cap.read()
            #print('프레임 시작')
            if ret == True:
                #cv2_imshow(frame)
                frame_cnt += 1

                if cv2.waitKey(25)  == ord('q'):
                    break

                #image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이지만 MediaPipe는 RGB형식
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image.flags.writeable = False
                results = pose.process(image)      # results.pose_landmarks 에 랜드마크 정보가 저장됨
                '''
                image.flags.writeable = True
                image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
                mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)  # 랜드마크 정보를 사용하여 연결선을 그림
                out.write(image)   # 랜드마크가 그려진 이미지를 파일에 저장
                '''
                if results.pose_landmarks==None:
                    print('frame',frame_cnt, None, entry.lower())
                    no_landmark_frames += 1
                    failed_file_names.append(entry.lower())
                else:
                    df_row = []

                    for idx in bone_idx:
                        # 한행에 컬럼을 추가
                        df_row.extend([round(results.pose_landmarks.landmark[idx].x,5),
                                    round(results.pose_landmarks.landmark[idx].y,5),
                                    round(results.pose_landmarks.landmark[idx].z,5)])

                    df.loc[len(df.index)] = df_row   # 한 행 추가

                #break  # 첫 프레임만 확인하려면 break 활성화
            else:   # 읽어온 영상 데이터가 없는 경우 반복문 종료
                break
        # End of while()

        print('총 프레임 수={}'.format(len(df)), 'landmark를 추출할 수 없는 프레임 수={}'.format(no_landmark_frames))

        cap.release()
    # End of for()   # 한개의 mp4 파일 처리 종료됨

    df.to_csv('/content/drive/MyDrive/Thesis/Anim_videos/Real_Videos/csv_run/'+os.path.splitext(entry)[0]+'.csv', header=True, index=False)
#out.release()
cv2.destroyAllWindows()
#df.to_csv('/content/drive/MyDrive/Thesis/mediapipe_run_all_ang_lohi_all_rev_lohi_landmarks.csv', header=True, index=False)
display(failed_file_names)
#df

## LSTM 모델을 사용한 Landmark 학습/분류 모델 생성
* Landmark가 저장된 디렉토리(walk/run)를 지정하고 코드 실행
* 실험 대상 Landmark
  + 실제 비디오 Landmark
  + Blender Anim video Landmark
  + 증강된(모델이 생성한) Landmark

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Attention, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# 데이터 로드 및 전처리 함수
import os
import pandas as pd
import numpy as np

# CUDA 비결정론적 동작 방지
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# 쓰레드 수 고정
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'


import tensorflow as tf
from tensorflow.keras import backend as K

# GPU의 결정적 연산 활성화
tf.config.experimental.enable_op_determinism()

SEED = 19

# Python random seed 설정
random.seed(SEED)

# NumPy random seed 설정
np.random.seed(SEED)

# TensorFlow random seed 설정
tf.random.set_seed(SEED)

def load_and_preprocess_data(folder_path, sequence_length):
    data = []
    labels = []

    # 폴더 내 모든 파일 반복
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)  # 파일 경로 생성

        # CSV 파일만 처리
        if file.endswith('.csv'):  # 확장자가 .csv인지 확인
            try:
                # CSV 파일 읽기
                df = pd.read_csv(file_path)

                # '_z'가 포함된 컬럼 제거
                df = df.loc[:, ~df.columns.str.contains('_z')]

                # 슬라이딩 윈도우 방식으로 시퀀스 생성
                for i in range(0, len(df) - sequence_length + 1):
                    sequence = df.iloc[i:i + sequence_length].to_numpy()
                    data.append(sequence)
                    labels.append(0 if 'walk' in folder_path else 1)  # 경로에 따라 라벨 할당
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
                continue

    return np.array(data), np.array(labels)

# 데이터 경로 설정
walk_path = "/content/drive/MyDrive/Thesis/Anim_videos/CSV_Aug_Real_Mix/walk/"
run_path =  "/content/drive/MyDrive/Thesis/Anim_videos/CSV_Aug_Real_Mix/run/"

# 시퀀스 길이 설정
sequence_length = 30

# 데이터 로드 및 전처리
walk_data, walk_labels = load_and_preprocess_data(walk_path, sequence_length)
run_data, run_labels = load_and_preprocess_data(run_path, sequence_length)

# 데이터 결합
data = np.concatenate([walk_data, run_data], axis=0)
labels = np.concatenate([walk_labels, run_labels], axis=0)

# 데이터 셔플
indices = np.arange(len(data))
np.random.shuffle(indices)
data = data[indices]
labels = labels[indices]

# 데이터 분리 (80% 학습, 20% 테스트)
split_idx = int(len(data) * 0.8)
x_train, x_test = data[:split_idx], data[split_idx:]
y_train, y_test = labels[:split_idx], labels[split_idx:]

# LSTM 모델 구성
model = Sequential([
    LSTM(128, input_shape=(sequence_length, data.shape[2]), return_sequences=True, kernel_initializer=GlorotUniform(seed=SEED)),
    Dropout(0.2),
    LSTM(64, kernel_initializer=GlorotUniform(seed=SEED)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # 이진 분류
])

# 모델 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 콜백 설정
early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
model_checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/Thesis/Anim_videos/CSV_Aug_Real_Mix/aug_video_LSTM.keras",
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

# 모델 학습
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=1000,
    batch_size=32,
    callbacks=[early_stop, model_checkpoint]
)

# 모델 평가
eval_result = model.evaluate(x_test, y_test)
print(f"Test Loss: {eval_result[0]}, Test Accuracy: {eval_result[1]}")

# 학습 내역 시각화
plt.figure(figsize=(12, 4))

# Loss 시각화
plt.figure(figsize=(5, 3))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Loss & Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Loss, Accuracy')
plt.legend()
plt.show()

### 위에서 실제 비디오 데이터를 학습한 모델을 사용하여 실제 비디오 분류 실험
* 비디오 경로 : /content/drive/MyDrive/Thesis/RealVideos/test/untested/edited/
* 위에서 학습된 실제 비디오를 학습한 LSTM 모델 로드

In [ ]:
# 실제 비디오 데이터를 학습한 LSTM 모델 로드
from tensorflow.keras.models import load_model

# 저장된 모델 로드
model_path = "/content/drive/MyDrive/Thesis/Anim_videos/CSV_Aug_Real_Mix/real_video_set01_LSTM.keras"
loaded_model = load_model(model_path)

# 모델 평가
test_loss, test_accuracy = loaded_model.evaluate(x_test, y_test)
print(f"Loaded Model Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")


* 데이터 증강을 적용하여 학습된 LSTM 모델 로드

In [ ]:
# 실제 비디오 데이터에 데이터 증강을 위해 생성된 데이터를 추가하여 학습한 LSTM 모델 로드
from tensorflow.keras.models import load_model

# 저장된 모델 로드
model_path = "/content/drive/MyDrive/Thesis/Anim_videos/CSV_Aug_Real_Mix/seed_A-11_lstm_model.keras"
loaded_model = load_model(model_path)
print(loaded_model.summary())
# 모델 평가
#test_loss, test_accuracy = loaded_model.evaluate(x_test, y_test)
#print(f"Loaded Model Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

Model: "sequential_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_40 (LSTM)                       │ (None, 30, 128)             │          78,336 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_42 (Dropout)                 │ (None, 30, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_41 (LSTM)                       │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_43 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_42 (Dense)                     │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_43 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 395,909 (1.51 MB)

 Trainable params: 131,969 (515.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 263,940 (1.01 MB)

None


## 위에서 로드한 LSTM 모델을 사용하여 다수 개의 비디오 분류(걷기/달리기) 실험
* 150개의 비디오를 "걷기/달리기"로 분류하기 성능측정

In [ ]:
import cv2
import os

from google.colab.patches import cv2_imshow
import mediapipe as mp
import numpy as np
import sys
import pandas as pd
from collections import deque

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)


video_dir = '/content/drive/MyDrive/Thesis/RealVideos/test/untested/edited'

correct_cnt = 0
wrong_cnt = 0


video_no = 0
dic_list = []  # 각 비디오 파일의 행동인식 결과를 한 행(dict형)으로 저장하기 위한 자료구조, [{},{},....]


def count_mp4_files(directory):
    count = 0
    for filename in os.listdir(directory):
        if filename.endswith('.mp4'):
            count += 1
    return count

video_cnt = count_mp4_files(video_dir)
print(f'There are {video_cnt} mp4 files in the directory.')
print('총 비디오 수:', video_cnt)

for entry in os.listdir(video_dir):
    if os.path.isfile(os.path.join(video_dir, entry)) and entry.lower().endswith(".mp4"):
        video_no += 1
        print(video_no,'.', entry)
        video_path = os.path.join(video_dir, entry)
        cap = cv2.VideoCapture(video_path)

        #cap = cv2.VideoCapture('/content/drive/MyDrive/Thesis/RealVideos/test/untested/edited/walk_girl_front_wide_field0001-0130.mp4')

        #check if the video capture is open
        if(cap.isOpened() == False):
            print("Error Opening Video Stream Or File")

        w = round(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = round(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        fourcc = cv2.VideoWriter_fourcc(*'DIVX')

        # 1프레임과 다음 프레임 사이의 간격 설정
        delay = round(1000/fps)

        # 영상 저장
        # cv2.VideoWriter 객체 생성, 속성값 입력
        #out = cv2.VideoWriter('/content/drive/MyDrive/Thesis/run_0001-0067_output.mp4', fourcc, 20.0, (w, h))    # 비디오파일 저장을 위한 스트림 생성

        # landmark를 csv 파일에 저장하기 위한 자료구조
        landmark_list = []  # 한개의 프레임에 포함된 관절좌표,visibility를 한행으로 표현(2차원 배열)

        bone_idx = [11,12,13,14,15,16,23,24,25,25,27,28]

        tmp_col = '''LS_x, LS_y, LS_z, RS_x, RS_y, RS_z, LFA_x, LFA_y, LFA_z, RFA_x, RFA_y, RFA_z, LH_x, LH_y, LH_z, RH_x, RH_y, RH_z,
                    LUL_x,  LUL_y, LUL_z, RUL_x, RUL_y, RUL_z, LL_x, LL_y, LL_z, RL_x, RL_y, RL_z, LF_x, LF_y, LF_z, RF_x, RF_y, RF_z'''
        columns = [ tok.strip() for tok in tmp_col.split(',')]

        df = pd.DataFrame( columns=columns)

        sequence = []

        sequence_dq = deque(maxlen=30)

        frame_cnt = 0

        zero = 0
        one = 0

        while(cap.isOpened()):
            ret, frame = cap.read()    #한개의 프레임 읽기
            #print('프레임 시작')
            if ret == True:
                #cv2_imshow(frame)
                frame_cnt += 1

                #if cv2.waitKey(25)  == ord('q'):
                #    break

                #image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이지만 MediaPipe는 RGB형식
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image.flags.writeable = False
                results = pose.process(image)      # results.pose_landmarks 에 랜드마크 정보가 저장됨
                image.flags.writeable = True
                image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
                #mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)  # 랜드마크 정보를 사용하여 연결선을 그림
                #out.write(image)   # 랜드마크가 그려진 이미지를 파일에 저장

                if results.pose_landmarks==None:
                    print('frame',frame_cnt, None)
                else:
                    row = []

                    for idx in bone_idx:    # 한개의 프레임에 포함된 모든 bone 오브젝트 순회
                        '''
                        frame_landmark.append((round(results.pose_landmarks.landmark[idx].x,5),
                                            round(results.pose_landmarks.landmark[idx].y,5),
                                            round(results.pose_landmarks.landmark[idx].z,5),
                                            round(results.pose_landmarks.landmark[idx].visibility,5)))'''
                        # 한행에 컬럼을 추가
                        row.extend([round(results.pose_landmarks.landmark[idx].x,5),
                                    round(results.pose_landmarks.landmark[idx].y,5)])

                    sequence_dq.append(row)   # 한 행(프레임) 추가

                    if len(sequence_dq) < 30: continue    # 시계열 데이터 시퀀스 크기만큼  데이터 누적(30프레임)

                    #print(np.array(list(sequence_dq)).shape)   # (30,24)
                    pred = loaded_model.predict(np.array(list(sequence_dq)).reshape(1,30,24))
                    pred_value = pred.flatten()[0]

                    #print(f'frame no({frame_cnt}):', end='')
                    #print(pred_value,', 달리기' if pred_value>0.5 else '걷기')

                    if pred_value>=0.5: one += 1
                    if pred_value<0.5: zero += 1



                #break  # 첫 프레임만 확인하려면 break 활성화
            else:
                break


        print('총 프레임 수={}'.format(frame_cnt), 'zero=',zero, 'one=',one)


        if entry.lower().endswith(".mp4") and (entry.find('walk')==0):
            if zero>one:
                correct_cnt += 1
                print('*** O ***', entry)
            else:
                wrong_cnt += 1
                print('*** X ***', entry)
        elif entry.lower().endswith(".mp4") and (entry.find('run')==0 or entry.find('jog')==0 or entry.find('sprint')==0):
            if one>zero:
                correct_cnt += 1
                print('*** O ***', entry)
            else:
                wrong_cnt += 1
                print('*** X ***', entry)
        else:
            print('이외의 경우:', entry)

        dic_list.append({'fname':entry, 'label': 0 if entry.find('walk')==0 else 1, '0':zero, '1':one, 'model_pred':0 if zero>one else 1})

        print(video_no,'. 정답수=',correct_cnt, ' 오답수=', wrong_cnt)
        print()
        cap.release()
        #out.release()

import pandas as pd
import time

df = pd.DataFrame(dic_list)
#df.to_csv('/content/drive/MyDrive/Thesis/RealVideos/test/untested/edited/pred'+str(time.time())+'.csv')  #  정확도를 테스트하기 위해 추정된 내용 기록

#cv2.destroyAllWindows()
print('video count=', video_cnt, 'correct_cnt=',correct_cnt, 'accuracy=', round(correct_cnt/video_cnt*100,2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
총 프레임 수=95 zero= 0 one= 65
*** O *** running_woman_back_left_below_0001-0095.mp4
150 . 정답수= 132  오답수= 18

video count= 150 cor